In [1]:
import pandas as pd

data = pd.read_parquet('data/obs_sbn_X05_full_mpc_desig.parquet', engine='pyarrow')
catalog_med = pd.read_csv('../medellin-vroc-latest.csv')

## Object classification from the Minor Planet Center API

For each unique `mpc_designation` we query the MPC `get-orb` API and read the `categorization` block of the orbit (`orbit_type_str`: Main Belt, Apollo, Amor, Aten, Atira, Jovian Trojan, Hungaria, Hilda, Mars Crosser, Jupiter Coupled, Distant Object, ...).

The API takes one designation per request (~0.7 s each), so the requests run in parallel threads and results are cached to `data/mpc_orbit_types.csv`. If the run is interrupted, re-running the cell only queries objects that are not already in the cache. With 16 threads it handles about 10 objects/s, which is roughly 8 h for all ~300k objects. Set `N_MAX` to work on a subset.

In [2]:
import os
import re
import time
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

MPC_ORB_URL = 'https://data.minorplanetcenter.net/api/get-orb'
CACHE_FILE = 'data/mpc_orbit_types.csv'
N_WORKERS = 16     # parallel requests
N_MAX = None       # e.g. 2000 to query only a subset; None = all objects
CHUNK = 1000       # save the cache every CHUNK objects


def query_key(desig):
    """'(122248) 2000 OE29' -> '122248' (numbered); otherwise the provisional designation."""
    m = re.match(r'^\((\d+)\)', desig)
    return m.group(1) if m else desig


def get_orbit_type(desig, retries=3):
    """Query the MPC get-orb API and return the categorization of one object."""
    out = {'mpc_designation': desig, 'orbit_type': None, 'object_type': None, 'orbit_type_int': None}
    for attempt in range(retries):
        try:
            r = requests.get(MPC_ORB_URL, json={'desig': query_key(desig)}, timeout=30)
            if r.status_code == 200:
                j = r.json()
                orbs = j[0].get('mpc_orb', []) if isinstance(j, list) and j else []
                if orbs:
                    cat = orbs[0].get('categorization', {}) or {}
                    out.update(orbit_type=cat.get('orbit_type_str'),
                               object_type=cat.get('object_type_str'),
                               orbit_type_int=cat.get('orbit_type_int'))
                return out
        except (requests.RequestException, ValueError):
            pass
        time.sleep(2 * (attempt + 1))  # back off before retrying
    return out


# Load previous results so the cell can be re-run to resume
if os.path.exists(CACHE_FILE):
    cache = pd.read_csv(CACHE_FILE)
else:
    cache = pd.DataFrame(columns=['mpc_designation', 'orbit_type', 'object_type', 'orbit_type_int'])

desigs = catalog_med['mpc_designation'].dropna().unique()
cached = set(cache['mpc_designation'])
todo = [d for d in desigs if d not in cached]
if N_MAX is not None:
    todo = todo[:N_MAX]
print(f'{len(desigs)} unique objects, {len(cache)} cached, {len(todo)} to query')

t0 = time.time()
for start in range(0, len(todo), CHUNK):
    batch = todo[start:start + CHUNK]
    with ThreadPoolExecutor(N_WORKERS) as ex:
        results = [f.result() for f in as_completed(ex.submit(get_orbit_type, d) for d in batch)]
    cache = pd.concat([cache, pd.DataFrame(results)], ignore_index=True)
    cache.to_csv(CACHE_FILE, index=False)
    done = start + len(batch)
    rate = done / (time.time() - t0)
    print(f'{done}/{len(todo)} done  ({rate:.1f} obj/s, ~{(len(todo) - done) / rate / 60:.0f} min left)')

524 unique objects, 0 cached, 524 to query
524/524 done  (5.6 obj/s, ~0 min left)


C:\Users\aguju\AppData\Local\Temp\ipykernel_24700\2893612218.py:59: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  cache = pd.concat([cache, pd.DataFrame(results)], ignore_index=True)


In [3]:
# Add the orbit type to every observation (map avoids copying the large dataframe)
types = cache.drop_duplicates('mpc_designation').set_index('mpc_designation')
catalog_med['orbit_type'] = catalog_med['mpc_designation'].map(types['orbit_type'])
catalog_med['object_type'] = catalog_med['mpc_designation'].map(types['object_type'])

# Number of distinct objects per class
objects = catalog_med.dropna(subset=['mpc_designation']).drop_duplicates('mpc_designation')
print(objects['orbit_type'].value_counts(dropna=False))

orbit_type
Main Belt        499
None              11
Other              6
Mars Crosser       4
Jovian Trojan      3
TNO                1
Name: count, dtype: int64


In [4]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# a-e and a-i distribution by MPC class. Objects beyond A_SPLIT (TNOs, distant objects)
# go to a narrower right-hand column so the inner Solar System is not compressed.
A_SPLIT = 10  # au

# Class label: the MPC orbit type; if missing, fall back to the object type (e.g. comets)
cls = objects['orbit_type'].replace('None', pd.NA).fillna(objects['object_type']).fillna('Unknown')
plot_df = objects.assign(cls=cls)
counts = plot_df['cls'].value_counts()

symbols = {'Apollo': 'square', 'Aten': 'square-open', 'Atira': 'square-dot', 'Amor': 'triangle-up',
           'Mars Crosser': 'star', 'Hungaria': 'diamond-open', 'Main Belt': 'circle', 'Hilda': 'hexagon',
           'Jovian Trojan': 'diamond-tall', 'Jupiter Coupled': 'diamond', 'Other': 'x',
           'Centaur': 'cross', 'Neptune Trojan': 'pentagon', 'TNO': 'square', 'Unknown': 'circle-open'}
# Show classes from the inner to the outer Solar System (ordered by median a)
order = plot_df.groupby('cls')['a'].median().sort_values().index
palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2',
           '#7f7f7f', '#bcbd22', '#17becf']
colors_cls = {c: palette[i % len(palette)] for i, c in enumerate(order)}

fig = make_subplots(rows=2, cols=2, shared_xaxes='columns', shared_yaxes='rows',
                    column_widths=[0.7, 0.3], horizontal_spacing=0, vertical_spacing=0)

for c in order:
    d = plot_df[plot_df['cls'] == c]
    small = c == 'Main Belt'  # many points: draw as small dots
    marker = dict(symbol=symbols.get(c, 'circle'), color=colors_cls[c], size=3 if small else 7,
                  opacity=0.6 if small else 1)
    first = True  # one legend entry per class
    for col, part in [(1, d[d['a'] < A_SPLIT]), (2, d[d['a'] >= A_SPLIT])]:
        if part.empty:
            continue
        for row, ycol in [(1, 'e'), (2, 'incl')]:
            fig.add_trace(go.Scatter(
                x=part['a'], y=part[ycol], mode='markers', marker=marker,
                name=f'{c} ({counts[c]})', legendgroup=c, showlegend=first,
                customdata=part['mpc_designation'],
                hovertemplate=f'%{{customdata}}<br>a = %{{x:.3f}} au<br>{ycol} = %{{y:.3f}}<extra>{c}</extra>'),
                row=row, col=col)
            first = False

fig.update_xaxes(showline=True, mirror=True, ticks='outside')
fig.update_yaxes(showline=True, mirror=True, ticks='outside')
fig.update_xaxes(title_text='a (au)', row=2)
fig.update_yaxes(title_text='e', row=1, col=1)
fig.update_yaxes(title_text='i (deg)', row=2, col=1)
fig.update_layout(template='simple_white', width=1000, height=650,
                  legend=dict(title='MPC orbit type', itemsizing='constant'))
fig.show()

In [12]:
os.makedirs('figures', exist_ok=True)
fig.write_html('figures/orbital_distribution_a_e_i.html', include_plotlyjs='cdn')
# For a static PNG (plotly 6 needs kaleido >= 1.0: pip install -U kaleido):
# fig.write_image('figures/orbital_distribution_a_e_i.png', scale=2)